# VisProbe: Adversarial Training Effectiveness (500 Samples)

**⚡ GPU Runtime Required:** `Runtime > Change runtime type > T4 GPU`

**📊 Experiment:** Vanilla vs Adversarially-Trained ResNet50 on ImageNet

**⏱️ Expected Time:** ~1-2 hours on T4 GPU

**📝 Tests:**
1. Natural perturbations (Gaussian Blur)
2. Adversarial attacks (PGD)
3. **Compositional attacks** (Natural + Adversarial)

---

## Step 1: Install Dependencies (~2-3 minutes)

In [ ]:
%%time
print("Installing dependencies...\n")

!pip install -q torch torchvision
!pip install -q robustbench
!pip install -q adversarial-robustness-toolbox
!pip install -q git+https://github.com/fra31/auto-attack

# Install VisProbe (UPDATE THIS if your repo is public)
# Option A: From GitHub
!pip install -q git+https://github.com/bilgedemirkaya/VisProbe

# Option B: Upload as zip (uncomment if needed)
# from google.colab import files
# uploaded = files.upload()
# !unzip -q VisProbe.zip
# !pip install -q -e VisProbe/

print("\n✅ All dependencies installed!")

## Step 2: Mount Google Drive and Set ImageNet Path (~30 seconds)

**⚠️ ACTION REQUIRED:** Update `IMAGENET_PATH` to match your Google Drive folder!

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# ==========================================
# ⚠️ UPDATE THIS PATH!
# ==========================================
IMAGENET_PATH = '/content/drive/MyDrive/imagenet_val'

# Common alternatives:
# IMAGENET_PATH = '/content/drive/MyDrive/datasets/imagenet_val'
# IMAGENET_PATH = '/content/drive/MyDrive/data/imagenet/val'

# Verify path
if os.path.exists(IMAGENET_PATH):
    num_classes = len([d for d in os.listdir(IMAGENET_PATH) 
                       if os.path.isdir(os.path.join(IMAGENET_PATH, d))])
    print(f"✅ ImageNet found at: {IMAGENET_PATH}")
    print(f"   Number of classes: {num_classes}")
    if num_classes != 1000:
        print(f"   ⚠️  WARNING: Expected 1000 classes, found {num_classes}")
else:
    print(f"❌ ERROR: ImageNet not found at: {IMAGENET_PATH}")
    print("   Please update IMAGENET_PATH to your ImageNet validation folder!")
    raise FileNotFoundError(IMAGENET_PATH)

## Step 3: Setup (~30 seconds)

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import json
from datetime import datetime

# Configuration
NUM_SAMPLES = 500  # Paper experiments
BATCH_SIZE = 32    # Reduce to 16 or 8 if OOM errors

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: GPU not available! This will be VERY slow (6-12 hours).")
    print("   Enable GPU: Runtime > Change runtime type > T4 GPU")

# ImageNet transforms
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load dataset
print("\nLoading ImageNet validation set...")
imagenet_val = ImageFolder(root=IMAGENET_PATH, transform=transform)
print(f"✅ Loaded {len(imagenet_val)} images from {len(imagenet_val.classes)} classes")

## Step 4: Load Models (~1-2 minutes)

In [ ]:
from robustbench.utils import load_model

print("Loading models...\n")

# Model 1: Vanilla ResNet50
print("1/2: Loading vanilla ResNet50...")
vanilla = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
vanilla = vanilla.to(device).eval()
print("    ✅ Vanilla model loaded")

# Model 2: Adversarially-trained from RobustBench
print("\n2/2: Loading adversarially-trained model (Salman2020Do_R50)...")
robust = load_model(
    model_name='Salman2020Do_R50',
    dataset='imagenet',
    threat_model='Linf'
)
robust = robust.to(device).eval()
print("    ✅ Robust model loaded")

print("\n✅ Both models ready!")

## Step 5: Get Mutually Correct Samples (~5-10 minutes)

Finding samples that both models classify correctly (to ensure fair comparison).

In [ ]:
%%time

def get_mutually_correct_samples(model1, model2, dataset, n=500):
    """Get samples correctly classified by BOTH models."""
    model1.eval()
    model2.eval()
    
    samples = []
    
    print(f"Searching for {n} samples correctly classified by both models...")
    print("(This may take 5-10 minutes)\n")
    
    with torch.no_grad():
        for i in range(len(dataset)):
            if len(samples) >= n:
                break
            
            img, label = dataset[i]
            img_batch = img.unsqueeze(0).to(device)
            
            pred1 = model1(img_batch).argmax().item()
            pred2 = model2(img_batch).argmax().item()
            
            if pred1 == label and pred2 == label:
                samples.append((img, label))
            
            if (i + 1) % 500 == 0:
                print(f"  Scanned {i+1:5d} images | Found {len(samples):3d} mutual correct | "
                      f"Rate: {len(samples)/(i+1)*100:.1f}%")
    
    return samples

samples = get_mutually_correct_samples(vanilla, robust, imagenet_val, n=NUM_SAMPLES)
print(f"\n✅ Found {len(samples)} mutually correct samples")
print(f"   Using these for all robustness tests\n")

## Step 6: Run ALL Robustness Tests

### Test 1: Natural Perturbations (~5-10 minutes)

In [ ]:
%%time
from visprobe import search

results = {"vanilla": {}, "robust": {}}

print("=" * 60)
print("TEST 1: NATURAL PERTURBATIONS (Gaussian Blur)")
print("=" * 60)

for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
    print(f"\nTesting {model_name}...")
    report = search(
        model=model,
        data=samples,
        perturbation="gaussian_blur",
        level_lo=0.0,
        level_hi=10.0,
        device=device,
        normalization="imagenet"
    )
    results[model_name]["natural"] = report
    print(f"  ✅ {model_name}: Pass rate = {report.score:.1f}%")

### Test 2: Adversarial Attacks (~20-40 minutes)

⚠️ **This is the slowest test!** The robust model searches more thoroughly.

In [ ]:
%%time
from visprobe.strategies import PGDStrategy

print("\n" + "=" * 60)
print("TEST 2: ADVERSARIAL ATTACKS (PGD)")
print("This is what adversarial training was designed for")
print("=" * 60)

for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
    print(f"\nTesting {model_name}...")
    print(f"  (This may take 10-20 minutes for {model_name})")
    
    report = search(
        model=model,
        data=samples,
        strategy=lambda eps: PGDStrategy(eps=eps, eps_step=max(eps/10, 1e-6), max_iter=20),
        level_lo=0.0,
        level_hi=0.03,
        device=device,
        normalization="imagenet",
        strategy_name="PGD Attack"
    )
    results[model_name]["adversarial"] = report
    print(f"  ✅ {model_name}: Pass rate = {report.score:.1f}%")

### Test 3: Compositional Attacks (~30-60 minutes)

**🔬 THE KEY CONTRIBUTION:** Natural perturbation + Adversarial attack combined

In [ ]:
%%time
from visprobe.strategies import (
    GaussianBlurStrategy,
    BrightnessStrategy,
    GaussianNoiseStrategy,
    Compose
)

print("\n" + "=" * 60)
print("TEST 3: COMPOSITIONAL ATTACKS (Novel Threat Model)")
print("Environmental + Adversarial combined")
print("=" * 60)

scenarios = [
    (
        "Low-Light PGD",
        lambda s: Compose([
            BrightnessStrategy(brightness_factor=0.3 + 0.7 * (1 - s)),
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
    (
        "Blurred PGD",
        lambda s: Compose([
            GaussianBlurStrategy(sigma=s * 2.0),
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
    (
        "Noisy PGD",
        lambda s: Compose([
            GaussianNoiseStrategy(std_dev=s * 0.03),
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
]

for scenario_name, composition in scenarios:
    print(f"\n{scenario_name}:")
    
    for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
        report = search(
            model=model,
            data=samples,
            strategy=composition,
            level_lo=0.0,
            level_hi=1.0,
            device=device,
            normalization="imagenet",
            strategy_name=scenario_name
        )
        results[model_name][scenario_name] = report
        print(f"  ✅ {model_name}: Pass rate = {report.score:.1f}%")

print("\n✅ All tests complete!")

## Step 7: Compute Comprehensive Results

In [ ]:
# Max thresholds for normalization
max_thresholds = {
    "natural": 10.0,
    "adversarial": 0.03,
    "Low-Light PGD": 1.0,
    "Blurred PGD": 1.0,
    "Noisy PGD": 1.0,
}

def get_threshold(report):
    """Extract failure threshold from report."""
    if hasattr(report, 'metrics') and report.metrics:
        return report.metrics.get('failure_threshold', 0)
    return 0

def normalized_robustness(threshold, max_thresh):
    """Calculate normalized robustness (0-100%)."""
    return min(100.0, (threshold / max_thresh) * 100) if max_thresh > 0 else 0

# Build comprehensive results
comprehensive_results = {
    "experiment": {
        "name": "Adversarial Training Effectiveness",
        "dataset": "ImageNet",
        "num_samples": len(samples),
        "device": device,
        "models": {
            "vanilla": "ResNet50 (IMAGENET1K_V2)",
            "robust": "Salman2020Do_R50 (RobustBench, Linf)"
        }
    },
    "tests": {},
    "timestamp": datetime.now().isoformat()
}

print("\n" + "=" * 85)
print("ADVERSARIAL TRAINING EFFECTIVENESS - FINAL RESULTS")
print("=" * 85)
print(f"Dataset: ImageNet | Samples: {len(samples)} | Device: {device}")
print("\nMetric: Normalized Robustness (threshold / max_threshold × 100%)")
print("Higher = more robust (tolerates larger perturbations)\n")

print("Threat Model          | Vanilla |  Robust | Improvement | Thresholds")
print("-" * 85)

for test_name in ["natural", "adversarial", "Low-Light PGD", "Blurred PGD", "Noisy PGD"]:
    v_report = results["vanilla"][test_name]
    r_report = results["robust"][test_name]
    
    v_thresh = get_threshold(v_report)
    r_thresh = get_threshold(r_report)
    max_thresh = max_thresholds[test_name]
    
    v_robust = normalized_robustness(v_thresh, max_thresh)
    r_robust = normalized_robustness(r_thresh, max_thresh)
    improvement = r_robust - v_robust
    
    display_name = "Natural (Blur)" if test_name == "natural" else \
                  "Adversarial (PGD)" if test_name == "adversarial" else test_name
    
    # Store in comprehensive results
    comprehensive_results["tests"][test_name] = {
        "display_name": display_name,
        "max_threshold": max_thresh,
        "vanilla": {
            "threshold": v_thresh,
            "normalized_robustness": round(v_robust, 2),
            "pass_rate": round(v_report.score, 2),
            "passed_samples": v_report.passed_samples,
            "failed_samples": v_report.failed_samples,
            "total_samples": v_report.total_samples,
            "runtime": round(v_report.runtime, 2)
        },
        "robust": {
            "threshold": r_thresh,
            "normalized_robustness": round(r_robust, 2),
            "pass_rate": round(r_report.score, 2),
            "passed_samples": r_report.passed_samples,
            "failed_samples": r_report.failed_samples,
            "total_samples": r_report.total_samples,
            "runtime": round(r_report.runtime, 2)
        },
        "improvement": round(improvement, 2),
        "improvement_ratio": round(r_thresh / v_thresh, 2) if v_thresh > 0 else None
    }
    
    emoji = "✓" if improvement >= 10 else "✗"
    thresh_str = f"v:{v_thresh:.4f} r:{r_thresh:.4f}"
    print(f"{display_name:22} | {v_robust:6.1f}% | {r_robust:6.1f}% |   {improvement:+6.1f}% {emoji} | {thresh_str}")

# Summary
adv_improvement = comprehensive_results["tests"]["adversarial"]["improvement"]
comp_improvements = [comprehensive_results["tests"][t]["improvement"]
                    for t in ["Low-Light PGD", "Blurred PGD", "Noisy PGD"]]
avg_comp_improvement = sum(comp_improvements) / len(comp_improvements)

print("\n" + "=" * 85)
print("KEY FINDINGS:")
print("-" * 85)
print(f"  • Adversarial (PGD) improvement:     {adv_improvement:+.1f}%")
print(f"  • Compositional average improvement: {avg_comp_improvement:+.1f}%")
print(f"  • Protection Gap: {adv_improvement - avg_comp_improvement:.1f}% less protection on compositional")
print()
print("  📝 CONCLUSION: Adversarial training provides strong protection against")
print("     pure PGD attacks, but LIMITED protection against compositional attacks.")
print("     Standard adversarial training is INSUFFICIENT for real-world robustness!")
print("=" * 85)

comprehensive_results["summary"] = {
    "adversarial_improvement": round(adv_improvement, 2),
    "compositional_avg_improvement": round(avg_comp_improvement, 2),
    "protection_gap": round(adv_improvement - avg_comp_improvement, 2),
    "conclusion": "Adversarial training provides strong protection against pure PGD but limited protection against compositional attacks"
}

## Step 8: Save and Download Results

In [ ]:
from google.colab import files

# Save comprehensive JSON
filename = f'experiment_results_{len(samples)}_samples.json'
with open(filename, 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

print(f"\n✅ Results saved: {filename}")
print("\n📥 Downloading...")
files.download(filename)
print("\n✅ Download complete!")

## Step 9: Generate Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract data
categories = ['Natural\n(Blur)', 'Adversarial\n(PGD)', 'Low-Light\nPGD', 'Blurred\nPGD', 'Noisy\nPGD']
test_keys = ['natural', 'adversarial', 'Low-Light PGD', 'Blurred PGD', 'Noisy PGD']

vanilla_scores = [comprehensive_results['tests'][k]['vanilla']['normalized_robustness'] for k in test_keys]
robust_scores = [comprehensive_results['tests'][k]['robust']['normalized_robustness'] for k in test_keys]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width/2, vanilla_scores, width, label='Vanilla ResNet50', 
               color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=1.5)
bars2 = ax.bar(x + width/2, robust_scores, width, label='Adversarially-Trained (Salman2020)', 
               color='#27ae60', alpha=0.8, edgecolor='black', linewidth=1.5)

ax.set_ylabel('Normalized Robustness (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Threat Model', fontsize=13, fontweight='bold')
ax.set_title('Adversarial Training Effectiveness on ImageNet\n'
             f'(N={len(samples)} samples, Higher = More Robust)', 
             fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(loc='upper left', fontsize=11)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", 
                ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", 
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add dividing line
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.7, linewidth=2)
ax.text(0.5, 105, 'Standard Threats', ha='center', fontsize=12, 
        color='#555', fontweight='bold', style='italic')
ax.text(3, 105, 'Compositional Threats (Novel)', ha='center', fontsize=12, 
        color='#555', fontweight='bold', style='italic')

# Add improvement annotations
for i, (v, r) in enumerate(zip(vanilla_scores, robust_scores)):
    improvement = r - v
    color = '#27ae60' if improvement > 10 else '#e74c3c'
    ax.annotate(f'{improvement:+.0f}%', xy=(i, max(v, r) + 10),
                ha='center', fontsize=10, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                         edgecolor=color, linewidth=2))

plt.tight_layout()
plt.savefig('adversarial_training_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📥 Downloading visualization...")
files.download('adversarial_training_effectiveness.png')
print("✅ Visualization downloaded!")

## Step 10: Print Summary for Copy-Paste

In [ ]:
print("\n" + "=" * 85)
print("📋 SUMMARY FOR PAPER")
print("=" * 85)

print("\n### Experimental Setup:")
print(f"- Dataset: ImageNet validation set")
print(f"- Samples: {len(samples)} (mutually correctly classified)")
print(f"- Models: ResNet-50 (vanilla) vs Salman2020Do_R50 (adversarially-trained)")
print(f"- Device: {device}")

print("\n### Key Results:")
for test_name, display_name in [("adversarial", "Pure PGD"), 
                                 ("Low-Light PGD", "Low-Light + PGD"),
                                 ("Blurred PGD", "Blur + PGD"),
                                 ("Noisy PGD", "Noise + PGD")]:
    data = comprehensive_results["tests"][test_name]
    v_rob = data["vanilla"]["normalized_robustness"]
    r_rob = data["robust"]["normalized_robustness"]
    imp = data["improvement"]
    ratio = data["improvement_ratio"]
    
    print(f"\n{display_name}:")
    print(f"  Vanilla:  {v_rob:.1f}%")
    print(f"  Robust:   {r_rob:.1f}%")
    print(f"  Improvement: {imp:+.1f}% ({ratio:.1f}× better)" if ratio else f"  Improvement: {imp:+.1f}%")

print("\n### Protection Gap:")
print(f"  Adversarial training improves PGD robustness by {adv_improvement:+.1f}%")
print(f"  But compositional robustness only by {avg_comp_improvement:+.1f}%")
print(f"  Protection gap: {comprehensive_results['summary']['protection_gap']:.1f} percentage points")

print("\n" + "=" * 85)
print("\n✅ ALL DONE! Download your results and visualizations above.")
print("\n📊 Files ready for your paper:")
print(f"   1. {filename}")
print("   2. adversarial_training_effectiveness.png")
print("=" * 85)